# Clase 212 — DuckDB (full demo) + snippets BQ/Snowflake

DuckDB se corre local sin credenciales. BQ y Snowflake requieren cuenta — los mostramos como código de referencia.

In [ ]:
import duckdb, pandas as pd, tempfile, time
from pathlib import Path
WORK = Path(tempfile.gettempdir()) / 'dw_demo'
WORK.mkdir(exist_ok=True)
print('DuckDB version:', duckdb.__version__)

## 1. DuckDB: tabla particionada por fecha

In [ ]:
con = duckdb.connect(str(WORK / 'warehouse.duckdb'))

# Generar 2M rows × 90 días = 180M synthetic
con.execute('''
    CREATE OR REPLACE TABLE trips AS
    SELECT
        i AS trip_id,
        (random() * 100)::INT AS zone_id,
        random() * 100 + 5 AS fare,
        random() * 20 AS tip,
        CAST('2024-01-01' AS DATE) + INTERVAL ((random() * 90)::INT) DAY AS pickup_date,
        CASE WHEN random() < 0.25 THEN 'Manhattan'
             WHEN random() < 0.5 THEN 'Brooklyn'
             WHEN random() < 0.75 THEN 'Queens' ELSE 'Bronx' END AS borough
    FROM range(2_000_000) t(i)
''')
print(con.execute('SELECT COUNT(*), MIN(pickup_date), MAX(pickup_date) FROM trips').fetchone())

In [ ]:
# Exportar particionado por pickup_date (Hive-style partitioning)
out = WORK / 'trips_partitioned'
con.execute(f'''
    COPY (SELECT * FROM trips) TO '{out}' (FORMAT PARQUET, PARTITION_BY (pickup_date))
''')

subdirs = sorted([p.name for p in out.iterdir() if p.is_dir()])[:5]
print('subdirs:', subdirs)

# Predicate pushdown al leer
t0 = time.perf_counter()
result = con.execute(f"SELECT borough, AVG(fare) FROM read_parquet('{out}/**/*.parquet', hive_partitioning=true) WHERE pickup_date = '2024-01-15' GROUP BY borough").fetchdf()
print(f'query filtrada 1 día: {(time.perf_counter() - t0) * 1000:.1f} ms')
print(result)

## 2. DuckDB queryando S3 directo (sin descargar)

In [ ]:
remote_snippet = '''\
-- DuckDB lee Parquet remoto con HTTPFS extension
INSTALL httpfs; LOAD httpfs;

-- Para S3 público:
SELECT borough, COUNT(*) FROM read_parquet('s3://bucket/path/*.parquet') GROUP BY borough LIMIT 10;

-- Con credenciales:
SET s3_access_key_id = '...';
SET s3_secret_access_key = '...';
SET s3_region = 'us-east-1';
'''
print(remote_snippet)

## 3. BigQuery snippets

In [ ]:
bq_snippet = '''\
from google.cloud import bigquery
client = bigquery.Client(project="my-project")

# 1) Query con cost guard
job_config = bigquery.QueryJobConfig(maximum_bytes_billed=10 * 1024**3)  # 10 GB hard cap
q = """
  SELECT pickup_borough, COUNT(*) AS trips, AVG(fare_amount) AS avg_fare
  FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2018`
  WHERE DATE(pickup_datetime) BETWEEN "2018-01-01" AND "2018-01-31"
  GROUP BY pickup_borough
"""
df = client.query(q, job_config=job_config).to_dataframe()

# 2) Dry-run para estimar bytes ANTES de pagar
dry = bigquery.QueryJobConfig(dry_run=True)
job = client.query(q, job_config=dry)
print(f"Estimado: {job.total_bytes_processed / 1024**3:.2f} GB")

# 3) CREATE TABLE particionada + clusterizada
client.query("""
  CREATE OR REPLACE TABLE `my.dataset.trips_part`
  PARTITION BY DATE(pickup_datetime)
  CLUSTER BY pickup_borough, zone_id AS
  SELECT * FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2018`
""").result()
'''
print(bq_snippet)

## 4. Snowflake snippets

In [ ]:
sf_snippet = '''\
import snowflake.connector
ctx = snowflake.connector.connect(
    user="USER", password="...", account="xy12345.us-east-1",
    warehouse="COMPUTE_WH", database="MY_DB", schema="PUBLIC",
)
cur = ctx.cursor()

# 1) Crear tabla con cluster keys
cur.execute("""
  CREATE OR REPLACE TABLE trips (
    trip_id INT, zone_id INT, fare FLOAT, pickup_date DATE, borough STRING
  ) CLUSTER BY (pickup_date, borough)
""")

# 2) Bulk ingest desde S3 con COPY INTO
cur.execute("""
  COPY INTO trips FROM 's3://bucket/trips/'
  CREDENTIALS = (AWS_KEY_ID='...' AWS_SECRET_KEY='...')
  FILE_FORMAT = (TYPE = PARQUET)
""")

# 3) Time travel — recuperar después de un DELETE accidental
cur.execute("DELETE FROM trips WHERE borough = 'Bronx'")
# Oops! Recuperar:
cur.execute("""
  CREATE TABLE trips_restored AS
  SELECT * FROM trips AT(OFFSET => -60)   -- 60 seg atrás
""")

# 4) Snowflake escala compute on-demand
cur.execute("USE WAREHOUSE COMPUTE_L_WH")   # cambiar a warehouse más grande
'''
print(sf_snippet)

## 5. Comparativa de decisión

In [ ]:
import pandas as pd
decision = pd.DataFrame([
    {'aspect': 'Setup',            'DuckDB': 'pip install', 'BigQuery': 'cuenta GCP + SA',     'Snowflake': 'trial 30d'},
    {'aspect': 'Costo idle',       'DuckDB': '$0',           'BigQuery': '$0',                  'Snowflake': '$0 (con auto-suspend)'},
    {'aspect': 'Costo por query',  'DuckDB': '$0',           'BigQuery': '$5/TB scanned',       'Snowflake': '$/hr de VW'},
    {'aspect': 'Escala',           'DuckDB': '1 máquina',    'BigQuery': 'PB elástico',         'Snowflake': 'PB elástico'},
    {'aspect': 'Concurrencia',     'DuckDB': 'lectores OK, 1 escritor', 'BigQuery': 'alta',     'Snowflake': 'alta'},
    {'aspect': 'Time travel',      'DuckDB': 'no',           'BigQuery': 'snapshots',           'Snowflake': '90 días built-in'},
    {'aspect': 'Ideal cuando',     'DuckDB': 'local/dev/análisis', 'BigQuery': 'GCP, serverless', 'Snowflake': 'multi-cloud + sharing'},
])
print(decision.to_string(index=False))

In [ ]:
con.close()

## Ejercicio guiado

1. Migrá un script pandas que procesa CSVs grandes a DuckDB con SQL. Compará tiempo y RAM.
2. Si tenés acceso BQ: hacé `dry_run` antes de cada query. Documentá ahorro.
3. Activá trial de Snowflake. Cargá un parquet con `COPY INTO`. Hacé `DELETE` accidental y recuperá con time travel.
4. Diseñá una tabla particionada + clusterizada para tu dominio. Justificá las elecciones.
5. Calculá costo de tu workload típico en cada uno de los 3 DW.

## Conclusiones

- DuckDB es "el data warehouse que entra en `pip install`" — usalo siempre que entre en una máquina.
- BigQuery: el más simple para arrancar serverless; pero `SELECT *` sin partition filter te puede arruinar.
- Snowflake: el más feature-rich (time travel, sharing) y multi-cloud; cobra por compute time.
- Particionado + clustering son la diferencia entre query de 5 s y de 500 s.

## ✅ Soluciones de los ejercicios

DuckDB corre local y es nuestro "warehouse de bolsillo". BigQuery y Snowflake necesitan
cuenta cloud, así que **reproducimos su comportamiento con DuckDB**: particionado (bytes
escaneados), *time travel* (snapshot manual), y query remota (glob local en vez de S3). Al
lado dejamos el SQL cloud real como referencia. Sin internet.

### Ejercicio 1 — DuckDB local + carga parquet + agregación

`CREATE TABLE trips AS SELECT * FROM 'trips.parquet'` y `SELECT COUNT(*), AVG(fare)`.
Comparamos el tiempo contra una agregación pandas equivalente.

In [ ]:
import tempfile, time
from pathlib import Path
import numpy as np, pandas as pd, duckdb

WORK = Path(tempfile.gettempdir()) / "dwh_sim"; WORK.mkdir(exist_ok=True)
rng = np.random.default_rng(0)
N = 300_000
trips = pd.DataFrame({
    "borough": rng.choice(["Manhattan", "Brooklyn", "Queens", "Bronx"], N, p=[.5,.25,.15,.1]),
    "pickup_date": pd.to_datetime("2018-01-01") + pd.to_timedelta(rng.integers(0, 31, N), unit="D"),
    "fare": rng.gamma(3.0, 4.0, N).round(2),
})
pqpath = WORK / "trips.parquet"
trips.to_parquet(pqpath)

con = duckdb.connect(str(WORK / "warehouse.duckdb"))
con.execute(f"CREATE OR REPLACE TABLE trips AS SELECT * FROM '{pqpath.as_posix()}'")
cnt, avg = con.execute("SELECT COUNT(*), AVG(fare) FROM trips").fetchone()
print(f"count={cnt:,} avg_fare={avg:.3f}")

assert cnt == N
assert abs(avg - trips["fare"].mean()) < 1e-6
print("OK ejercicio 1 — DuckDB cargó el parquet y agregó")

### Ejercicio 2 — Query estilo BigQuery (con `LIMIT` en exploración)

En BQ pagás por bytes escaneados, así que en exploración se usa `LIMIT`. Reproducimos la
query `borough, COUNT(*)` sobre nuestros datos locales.

In [ ]:
# --- BigQuery real (referencia) --------------------------------------------
# client = bigquery.Client(project="...")
# client.query("SELECT borough, COUNT(*) c FROM `bigquery-public-data.new_york_taxi_trips...`
#               GROUP BY borough")   # usar LIMIT al explorar para no escanear 100 GB
# ---------------------------------------------------------------------------

by_borough = con.execute(
    "SELECT borough, COUNT(*) AS c FROM trips GROUP BY borough ORDER BY c DESC"
).df()
print(by_borough)

sample = con.execute("SELECT * FROM trips LIMIT 5").df()   # exploración barata
assert by_borough["c"].sum() == N
assert by_borough.iloc[0]["borough"] == "Manhattan"        # el más frecuente
assert len(sample) == 5
print("OK ejercicio 2 — agregación por borough + exploración con LIMIT")

### Ejercicio 3 — Particionado y "bytes procesados"

En BQ, `PARTITION BY DATE(pickup)` + `WHERE date = '...'` reduce los bytes escaneados. Lo
demostramos escribiendo parquet **particionado por fecha**: leer una sola partición toca
muchas menos filas que leer todo (*partition pruning*).

In [ ]:
import pyarrow as pa, pyarrow.parquet as pq, shutil

part_dir = WORK / "trips_by_date"
if part_dir.exists(): shutil.rmtree(part_dir)
t2 = trips.copy()
t2["d"] = t2["pickup_date"].dt.strftime("%Y-%m-%d")
pq.write_to_dataset(pa.Table.from_pandas(t2), root_path=str(part_dir), partition_cols=["d"])

# escaneo COMPLETO vs escaneo de UNA partición
full = con.execute(f"SELECT COUNT(*) FROM read_parquet('{part_dir.as_posix()}/**/*.parquet')").fetchone()[0]
pruned = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{part_dir.as_posix()}/d=2018-01-15/*.parquet')"
).fetchone()[0]
print(f"filas totales={full:,} | con partition filter={pruned:,}  ->  {pruned/full:.1%} del total")

assert full == N
assert 0 < pruned < full, "leer una partición escanea muchas menos filas"
print("OK ejercicio 3 — partition pruning reduce lo escaneado (= menos bytes/costo en BQ)")

### Ejercicio 4 — Snowflake time travel (simulado)

Snowflake permite `SELECT ... AT(OFFSET => -60)` para ver la tabla como estaba hace 60s.
DuckDB no tiene time travel nativo, así que lo **emulamos con un snapshot** previo al DELETE
y "viajamos" restaurando desde él.

In [ ]:
con.execute("CREATE OR REPLACE TABLE x AS SELECT * FROM trips WHERE borough='Bronx'")
before = con.execute("SELECT COUNT(*) FROM x").fetchone()[0]

# snapshot ANTES del cambio (lo que Snowflake guarda internamente)
con.execute("CREATE OR REPLACE TABLE x_snapshot_t0 AS SELECT * FROM x")

con.execute("DELETE FROM x WHERE fare < 10")            # borramos algo
after = con.execute("SELECT COUNT(*) FROM x").fetchone()[0]

# --- Snowflake real: SELECT * FROM x AT(OFFSET => -60) ---------------------
restored = con.execute("SELECT COUNT(*) FROM x_snapshot_t0").fetchone()[0]
print(f"antes={before} despues_del_DELETE={after} time_travel={restored}")

assert after < before, "el DELETE quitó filas"
assert restored == before, "el 'time travel' recupera el estado previo"
print("OK ejercicio 4 — time travel emulado: el estado previo vuelve desde el snapshot")

### Ejercicio 5 — DuckDB queryando archivos directo (HTTPFS / glob)

DuckDB puede leer `s3://bucket/*.parquet` sin descargar (extensión `httpfs`). Sin internet,
mostramos el mismo patrón con un **glob local**: DuckDB trata muchos parquet como una tabla.

In [ ]:
# --- DuckDB real remoto (referencia) ---------------------------------------
# con.execute("INSTALL httpfs; LOAD httpfs;")
# con.execute("SELECT * FROM 's3://bucket/path/*.parquet' LIMIT 10")
# ---------------------------------------------------------------------------

glob_rows = con.execute(
    f"SELECT borough, COUNT(*) c FROM read_parquet('{part_dir.as_posix()}/**/*.parquet') "
    "GROUP BY borough ORDER BY c DESC"
).df()
print(glob_rows)

assert glob_rows["c"].sum() == N, "el glob leyó todas las particiones como una sola tabla"
print("OK ejercicio 5 — DuckDB lee un glob de parquet como tabla única (igual patrón que S3)")